# ForestGuard experiments on Colab
Runtime > Change runtime type > T4 GPU. Upload the `fg` folder and your data folder to Drive first.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive/fg
!pip -q install affine

In [ ]:
ROOT='/content/drive/MyDrive/forestguard_data'   # folder with manifest.csv + .npz patches
SEEDS=[0,1,2,3,4]
VARIANTS=['full','no_cma','early_only','late_only','optical_only','sar_only','unet_r34','unet_plain']

In [ ]:
!python split.py --root $ROOT --seed 0
!python split.py --root $ROOT --seed 0 --holdout_region Congo   # change to any region name in your manifest

In [ ]:
import subprocess
for v in VARIANTS:
    for s in SEEDS:
        subprocess.run(f'python train.py --root {ROOT} --split split_seed0.csv --variant {v} --seed {s} --epochs 40 --out runs', shell=True, check=True)
# transfer test (fewer variants)
for v in ['full','unet_r34','unet_plain']:
    for s in SEEDS:
        subprocess.run(f'python train.py --root {ROOT} --split split_seed0_holdout_Congo.csv --variant {v} --seed {s} --epochs 40 --out runs', shell=True, check=True)

In [ ]:
!python classical.py --root $ROOT --split split_seed0.csv

Run `python benchmark.py --threads 4` on the Dell CPU (not here), copy `benchmark.json` next to `runs/`, then:

In [ ]:
!python aggregate.py --runs runs --bench benchmark.json --out tables.md
print(open('tables.md').read())